In [4]:
from google.colab import drive
drive.mount('/content/drive')

import os
FOLDER = '/content/drive/MyDrive/CPI_Research'
os.makedirs(FOLDER, exist_ok=True)
print("✅ Drive mounted:", os.listdir(FOLDER))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive mounted: ['Leakage Ablation Runbook_session5.ipynb', 'bangladesh_MASTER_dataset.csv', 'lasso_selected_features.csv', 'session1_orders.json', 'fig_session1_predictions.png', 'fig_session2_predictions.png', 'FINAL_scoreboard.csv', 'fig_scoreboard.png', 'fig_eda_series.png', 'fig_correlation.png', 'FINAL_forecast_2026_2027.csv', 'FINAL_forecast_chart.png', 'all_model_results.csv', 'multiseed_results.csv', 'session5_leakage_ablation.py']


In [5]:
!pip install vmdpy -q

import pandas as pd
df = pd.read_csv(f"{FOLDER}/bangladesh_MASTER_dataset.csv")
print(df.shape)
print(df.columns.tolist())
print(df.head(3))

(317, 15)
['Date', 'CPI', 'CPI_Food', 'Inflation_YoY', 'ExchangeRate_BDT_USD', 'Forex_Reserves_USDmn', 'BroadMoney_BDTmn', 'Brent_Oil_USD', 'Fed_Funds_Rate', 'Gold_Price_Index', 'FAO_Food_Index', 'FAO_Cereals_Index', 'COVID_dummy', 'UkraineWar_dummy', 'BD_Unrest_dummy']
         Date      CPI  CPI_Food  Inflation_YoY  ExchangeRate_BDT_USD  \
0  2000-01-01  53.5541       NaN            NaN                  51.0   
1  2000-02-01  53.2376       NaN            NaN                  51.0   
2  2000-03-01  53.3055       NaN            NaN                  51.0   

   Forex_Reserves_USDmn  BroadMoney_BDTmn  Brent_Oil_USD  Fed_Funds_Rate  \
0             1568.5687               NaN        25.6333             NaN   
1             1614.5134               NaN        28.0305             NaN   
2             1586.2286               NaN        27.4943             NaN   

   Gold_Price_Index  FAO_Food_Index  FAO_Cereals_Index  COVID_dummy  \
0               NaN            52.8               52.1      

In [6]:
iy = df['Inflation_YoY']
print(iy.notna().sum(), df.loc[iy.first_valid_index(), 'Date'], iy.min())

185 2011-01-01 4.9594


In [7]:
from google.colab import files
files.upload()                                  # session5_leakage_ablation.py
!cp session5_leakage_ablation.py "{FOLDER}/"

!grep -E "^DATA_PATH|^DATE_COL|^CPI_COL" session5_leakage_ablation.py

Saving session5_leakage_ablation.py to session5_leakage_ablation.py
DATA_PATH = "/content/drive/MyDrive/CPI_Research/bangladesh_MASTER_dataset.csv"  # <-- SET FOLDER
DATE_COL  = "Date"                                                      # <-- SET
CPI_COL   = "CPI"                                                       # <-- SET


In [8]:
!cp session5_leakage_ablation.py "{FOLDER}/"
!ls -la "{FOLDER}/session5_leakage_ablation.py"

-rw------- 1 root root 14160 Jul 23 11:36 /content/drive/MyDrive/CPI_Research/session5_leakage_ablation.py


In [9]:
!python session5_leakage_ablation.py

2026-07-23 11:36:41.623914: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
observations=316  train=252  test=64
2026-07-23 11:36:52.099912: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1784806612.101376    5288 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
2026-07-23 11:36:55.470022: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91900

Model                   Decomposition         R2     RMSE      MAE    MAPE
SARIMA (refe

In [10]:
files.upload()          # session5b_warmup_diagnostic.py

Saving session5b_warmup_diagnostic.py to session5b_warmup_diagnostic.py


{'session5b_warmup_diagnostic.py': b'"""\nSESSION 5b - WARM-UP SENSITIVITY DIAGNOSTIC\n\nWHY THIS EXISTS\n---------------\nThe Session-5 causal VMD arm returned RMSE 3.4961 against the published\nTable VI value of 2.6349 (+32.7%). The wavelet arm, on the identical code\npath, reproduced to within 1.8%. The difference between the two arms is the\nwarm-up length: 24 months for wavelet, 48 for VMD.\n\nIf training length drives the deviation, causal VMD RMSE should move\nsystematically as the warm-up shrinks and the training window grows. If it\ndoes not move, the cause is the causal edge handling instead, and Session 3\nhas to be inspected directly.\n\nThis matters because the headline leakage premium is 47.5% against the\nSession-5 causal baseline but 30.3% against the published one, and we should\nnot publish either number until we know which baseline is right.\n\nRUN IN THE SAME COLAB SESSION, AFTER session5_leakage_ablation.py.\nTakes roughly as long as the VMD half of the main run, t

In [11]:
!cp session5b_warmup_diagnostic.py "{FOLDER}/"
!python session5b_warmup_diagnostic.py

warm-up sensitivity of the causal VMD arm
(full-series arm re-run at each warm-up so the premium stays like-for-like)

 warm-up  train mo.    causal  full-series   premium  vs Table VI
-----------------------------------------------------------------
2026-07-23 11:49:17.696098: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-07-23 11:49:23.591361: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1784807363.592863   18029 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capabili

In [2]:
import json, glob

FOLDER = '/content/drive/MyDrive/CPI_Research'

nbs = glob.glob(f"{FOLDER}/*.ipynb")
for n in nbs:
    print(n)

In [3]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [4]:
!find /content/drive/MyDrive -name "*session*3*.ipynb" 2>/dev/null

In [6]:
!find /content/drive/MyDrive -name "*.ipynb" 2>/dev/null

/content/drive/MyDrive/test1.ipynb
/content/drive/MyDrive/Colab Notebooks/ML test 1 .ipynb
/content/drive/MyDrive/Colab Notebooks/Deep learning 2.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled0.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled1.ipynb
/content/drive/MyDrive/Colab Notebooks/MASTER_IMPLEMENTATION_BANGLADESH_CPI.ipynb
/content/drive/MyDrive/Colab Notebooks/CPI Research session2.ipynb
/content/drive/MyDrive/Colab Notebooks/CPI_Research_session4_verification.ipynb
/content/drive/MyDrive/UnSupervised ML – Clustering.ipynb
/content/drive/MyDrive/Supervised ML-Regression.ipynb
/content/drive/MyDrive/Supervised ML-classification-1.ipynb
/content/drive/MyDrive/Supervised ML-classification-2.ipynb
/content/drive/MyDrive/CPI_Research/Leakage Ablation Runbook_session5.ipynb
/content/drive/MyDrive/CPI forecast research session 1.ipynb


In [7]:
import json
NB = "/content/drive/MyDrive/Colab Notebooks/MASTER_IMPLEMENTATION_BANGLADESH_CPI.ipynb"
nb = json.load(open(NB))
for i, c in enumerate(nb['cells']):
    if c['cell_type'] != 'code':
        continue
    src = ''.join(c['source'])
    if 'vmd' in src.lower():
        print(f"########## CELL {i} ##########")
        print(src)
        print()

########## CELL 5 ##########
# ═══════════════════════════════════════════════════════════════════
# CONFIGURATION — Set USE_PRECOMPUTED = False to train all models fresh
# ═══════════════════════════════════════════════════════════════════
USE_PRECOMPUTED = True    # True = fast review mode; False = full training (~2h on T4 GPU)

# ── Pre-computed results from the original Colab execution ──
PRECOMPUTED_SCOREBOARD = [
    {'Model': 'SARIMA(2,1,2)x(0,1,1,12)',     'R2': 0.9985, 'RMSE': 1.3053, 'MAE': 0.9370, 'MAPE': 0.384},
    {'Model': 'SARIMA-LSTM',                    'R2': 0.9985, 'RMSE': 1.3106, 'MAE': 0.9298, 'MAPE': 0.381},
    {'Model': 'SARIMA-HMM-LSTM',                'R2': 0.9983, 'RMSE': 1.3984, 'MAE': 0.9799, 'MAPE': 0.401},
    {'Model': 'Wavelet-SARIMA-LSTM',            'R2': 0.9977, 'RMSE': 1.6227, 'MAE': 1.2504, 'MAPE': 0.506},
    {'Model': 'ARIMA-LSTM',                     'R2': 0.9950, 'RMSE': 2.4047, 'MAE': 1.8126, 'MAPE': 0.743},
    {'Model': 'ARIMA-HMM-LSTM',   

In [8]:
FOLDER = '/content/drive/MyDrive/CPI_Research'
!pip install vmdpy -q

In [9]:
from google.colab import files
files.upload()          # session5c_ablation_matched.py

Saving session5c_ablation_matched.py to session5c_ablation_matched.py


{'session5c_ablation_matched.py': b'"""\nSESSION 5c - LEAKAGE ABLATION, REBUILT ON THE SESSION 3 PIPELINE\n\nWHY THIS SUPERSEDES SESSION 5\n-----------------------------\nSession 5 wrote its own downstream pipeline and its causal VMD arm scored\n3.4961 against the published 2.6349. Inspection of Session 3 showed the causal\ndecomposition there is strictly correct - VMD is called per month on vals[:i+1],\nu[:, -1] is stored, and the warm-up is dropped, never backfilled. There is no\nleak. The gap came from the downstream model, not the decomposition:\n\n    Session 3 trains the VMD-LSTM on RAW delta-CPI.\n    Session 5 standardised the target before training.\n\nSame loss and optimiser, different loss surface. So Session 5\'s causal arm was\na different (and weaker) model, which made its 47.5% premium unreliable.\n\nThis script fixes that by lifting the Session 3 code verbatim - same windowing,\nsame architecture, same unscaled VMD target, same clear_session discipline -\nand changing e

In [10]:
!cp session5c_ablation_matched.py "{FOLDER}/"
!python session5c_ablation_matched.py

2026-07-23 17:33:32.184609: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
observations=316  test=64
2026-07-23 17:33:38.931291: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1784828018.932818    6455 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
2026-07-23 17:33:42.171224: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91900
  Wavelet-SARIMA-LSTM   causal       n=64  RMSE=1.5928
  Wavelet-SARIMA-LSTM   full-series  n=64  R

In [11]:
from google.colab import files
files.upload()          # session5d_multiseed_ablation.py

Saving session5d_multiseed_ablation.py to session5d_multiseed_ablation.py


{'session5d_multiseed_ablation.py': b'"""\nSESSION 5d - MULTI-SEED LEAKAGE ABLATION\n\nWHY\n---\nSession 5c matched the Session 3 pipeline and produced a leakage premium of\n+10.4% (wavelet) and +28.6% (VMD). Two questions remain open:\n\n  1. Is the premium an effect or seed noise? A single seed cannot say.\n  2. The causal arms landed 1.8% and 6.5% below the published values. Is that\n     the GPU nondeterminism already documented in Section VIII, or something\n     systematic?\n\nBoth are answered by running the ablation over the SAME ten seeds the paper\nalready uses for its neural battery (Table IX): 42, 7, 13, 21, 99, 123, 256,\n314, 777, 2024. If the published value falls inside the causal seed\ndistribution, question 2 is closed. If the premium holds across seeds, so is 1.\n\nEFFICIENCY\n----------\nDecompositions and the SARIMA-on-smooth stage do not depend on the seed, so\nthey are computed once and reused across all ten runs. Only the LSTM stages\nare repeated.\n\nOUTPUT\n--

In [ ]:
!cp session5d_multiseed_ablation.py "{FOLDER}/"
!python session5d_multiseed_ablation.py

2026-07-23 17:43:09.855739: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
observations=316 test=64 seeds=10
precomputing decompositions (seed independent)...
done in 5s

2026-07-23 17:43:18.740770: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1784828598.742284   17881 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
2026-07-23 17:43:21.815314: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91900
  seed    42   caus-Wav=1.592